# Filter invalid tiles

Deletes tiles whose rasters fail `validate_tile_contents` or are missing any required files. Also drops their entries from `dataset/manifest.csv`.


In [1]:
from pathlib import Path
from typing import Iterable

import numpy as np
import pandas as pd
import rasterio

NOTEBOOK_DIR = Path.cwd()
DATASET_ROOT = NOTEBOOK_DIR / "dataset"
if not DATASET_ROOT.exists():
    alt_root = Path("1_download_data_and_create_dataset") / "dataset"
    if alt_root.exists():
        DATASET_ROOT = alt_root

RGB_INDEXES = (4, 3, 2)
MIN_RAW_VARIANCE = 1e-4
MIN_NORMALIZED_CHANNEL_STD = 0.02
MIN_SAMPLE_UNIQUE_COLORS = 16
UNIQUE_COLOR_SAMPLE_STRIDE = 8
MIN_PROBABILITY_STD = 0.005

REQUIRED_COLUMNS = (
    "sentinel_path",
    "dynamic_world_label_path",
    "dynamic_world_prob_path",
    "snow_mask_path",
)


def resolve_path(relative_path: str) -> Path:
    normalized = relative_path.replace('\\', '/')
    return DATASET_ROOT / Path(normalized)


def _ensure_exists(path: Path) -> Path:
    if not path.exists():
        raise FileNotFoundError(f"missing file: {path}")
    return path


def read_rgb_image(path: Path, rgb_indexes: Iterable[int] = RGB_INDEXES) -> np.ndarray:
    path = _ensure_exists(path)
    with rasterio.open(path) as src:
        valid_indexes = [idx for idx in rgb_indexes if idx <= src.count]
        if not valid_indexes:
            valid_indexes = [1]
        data = src.read(valid_indexes).astype(np.float32)
    data = np.moveaxis(data, 0, -1)
    data = np.clip(data, 0, None)

    if not np.isfinite(data).all():
        raise ValueError("raw Sentinel-2 image contains non-finite values")

    if data.ptp() < MIN_RAW_VARIANCE:
        raise ValueError("raw Sentinel-2 image is nearly constant")

    max_val = data.max()
    if max_val > 0:
        data /= max_val
    return data


def read_single_band_image(path: Path) -> np.ndarray:
    path = _ensure_exists(path)
    with rasterio.open(path) as src:
        data = src.read(1)
    return data


def summarize_probability_cube(path: Path) -> np.ndarray:
    path = _ensure_exists(path)
    with rasterio.open(path) as src:
        data = src.read().astype(np.float32)
    if data.ndim == 3 and data.shape[0] > 1:
        summary = data.max(axis=0)
    else:
        summary = data[0]
    summary_min, summary_max = np.min(summary), np.max(summary)
    if summary_max > summary_min:
        summary = (summary - summary_min) / (summary_max - summary_min)
    return summary


def validate_tile_contents(
    raw: np.ndarray, labels: np.ndarray, probs: np.ndarray, snow: np.ndarray
) -> None:
    height, width, _ = raw.shape
    expected_shape = (height, width)
    if labels.shape != expected_shape:
        raise ValueError(
            f"label raster has shape {labels.shape} but expected {expected_shape}"
        )
    if probs.shape != expected_shape:
        raise ValueError(
            f"probability raster has shape {probs.shape} but expected {expected_shape}"
        )
    if snow.shape != expected_shape:
        raise ValueError(
            f"snow mask has shape {snow.shape} but expected {expected_shape}"
        )

    flattened = raw.reshape(-1, raw.shape[-1])
    max_std = float(flattened.std(axis=0).max())
    if max_std < MIN_NORMALIZED_CHANNEL_STD:
        raise ValueError(
            f"raw Sentinel-2 image lacks contrast (max channel std {max_std:.4f})"
        )

    sampled = raw[::UNIQUE_COLOR_SAMPLE_STRIDE, ::UNIQUE_COLOR_SAMPLE_STRIDE]
    if sampled.size == 0:
        raise ValueError("raw Sentinel-2 image is empty")
    quantized = np.rint(np.clip(sampled, 0, 1) * 255).astype(np.uint8)
    flat_quantized = np.ascontiguousarray(quantized.reshape(-1, quantized.shape[-1]))
    if flat_quantized.size == 0:
        raise ValueError("raw Sentinel-2 image is empty")
    compound_dtype = np.dtype(
        [(f"c{i}", flat_quantized.dtype) for i in range(flat_quantized.shape[1])]
    )
    view = flat_quantized.view(compound_dtype)
    unique_colors = np.unique(view).size
    if unique_colors < MIN_SAMPLE_UNIQUE_COLORS:
        raise ValueError(
            f"raw Sentinel-2 image uses only {unique_colors} sampled colors (min {MIN_SAMPLE_UNIQUE_COLORS})"
        )

    prob_std = float(np.nanstd(probs))
    if not np.isfinite(prob_std) or prob_std < MIN_PROBABILITY_STD:
        raise ValueError(
            f"Dynamic World probability summary is uninformative (std {prob_std:.4f})"
        )


In [2]:
manifest_path = DATASET_ROOT / "manifest.csv"
if not manifest_path.exists():
    raise FileNotFoundError(f"Manifest not found at {manifest_path}")

manifest = pd.read_csv(manifest_path)
invalid_indices = []

for idx, row in manifest.iterrows():
    if idx % 100 == 0:
        print("Working on", idx, "/", len(manifest))
    tile_id = row["tile_id"]
    paths = {}
    try:
        for column in REQUIRED_COLUMNS:
            paths[column] = resolve_path(row[column])
        missing = [col for col, path in paths.items() if not path.exists()]
        if missing:
            missing_str = ", ".join(missing)
            raise FileNotFoundError(f"missing file(s) for {missing_str}")

        raw = read_rgb_image(paths["sentinel_path"])
        labels = read_single_band_image(paths["dynamic_world_label_path"])
        probs = summarize_probability_cube(paths["dynamic_world_prob_path"])
        snow = read_single_band_image(paths["snow_mask_path"])
        validate_tile_contents(raw, labels, probs, snow)
    except (ValueError, FileNotFoundError) as exc:
        print(f"Invalid tile {tile_id}: {exc}")
        invalid_indices.append(idx)
        for path in paths.values():
            if path.exists():
                path.unlink()

if invalid_indices:
    manifest = manifest.drop(index=invalid_indices)
    manifest.to_csv(manifest_path, index=False)
    print(f"Removed {len(invalid_indices)} tiles from manifest and deleted their rasters.")
else:
    print("All tiles passed validation; no files deleted.")

Working on 0 / 9385
Invalid tile tile_00000_coastal_marine_00001: raw Sentinel-2 image lacks contrast (max channel std 0.0075)
Invalid tile tile_00011_desert_steppe_00012: raw Sentinel-2 image lacks contrast (max channel std 0.0164)
Invalid tile tile_00044_coastal_marine_00045: raw Sentinel-2 image lacks contrast (max channel std 0.0144)
Invalid tile tile_00064_alpine_asia_00065: raw Sentinel-2 image lacks contrast (max channel std 0.0144)
Invalid tile tile_00077_coastal_marine_00078: raw Sentinel-2 image lacks contrast (max channel std 0.0124)
Invalid tile tile_00088_urban_global_00089: raw Sentinel-2 image lacks contrast (max channel std 0.0110)
Invalid tile tile_00101_snow_high_lat_00102: raw Sentinel-2 image lacks contrast (max channel std 0.0101)
Invalid tile tile_00102_snow_high_lat_00103: raw Sentinel-2 image lacks contrast (max channel std 0.0143)
Invalid tile tile_00120_snow_high_lat_00121: raw Sentinel-2 image lacks contrast (max channel std 0.0145)
Invalid tile tile_00144_ar